# Week 6 – Data Quality Checks | BingeMetrics

**Team:** Team 04  
**Project:** BingeMetrics – OTT & Music Engagement Analytics  
**Notebook:** `notebooks/04_data_quality_checks.ipynb`

This notebook evaluates the Week-5 Silver Candidate tables using the **eight approved Team 04 DQ rules**, retains all failures, routes each physical row exactly once to Trusted or Quarantine, and proves reconciliation.

**Approved rules:** `DQ-SES-001` to `DQ-SES-005`, `DQ-CNT-001`, `DQ-SUB-001`, `DQ-USR-001`.

> The approved DQ rulebook is the authority. Do not add rule IDs or invent approved dictionary values. Populate the configuration cells only where the rulebook/data dictionary supplies an approved value set.


## 1. Week 6 flow

Dependency order:

1. Content → Trusted Content / Quarantine Content
2. Users → Trusted Users / Quarantine Users
3. Subscriptions → Trusted Subscriptions / Quarantine Subscriptions
4. Sessions → Trusted Sessions / Quarantine Sessions

A physical Candidate row is never silently deleted. If several rules fail, all applicable rule IDs and reasons stay on the same Quarantine row.


In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;


active_catalog,active_schema
workspace,default


## 2. Confirm Week-5 Candidate inputs

This project previously created these Silver tables:

- `silver_users`
- `silver_subscriptions`
- `silver_sessions`
- `silver_content_catalog`

The DQ layer treats them as Candidate inputs.


In [0]:
%sql
SHOW TABLES;

SELECT 'silver_users' AS table_name, COUNT(*) AS row_count FROM silver_users
UNION ALL
SELECT 'silver_subscriptions', COUNT(*) FROM silver_subscriptions
UNION ALL
SELECT 'silver_sessions', COUNT(*) FROM silver_sessions
UNION ALL
SELECT 'silver_content_catalog', COUNT(*) FROM silver_content_catalog;


table_name,row_count
silver_users,24975
silver_subscriptions,34980
silver_sessions,249701
silver_content_catalog,2990


In [0]:
# Inspect actual Candidate schemas before evaluating rules.
for name in [
    "silver_content_catalog",
    "silver_users",
    "silver_subscriptions",
    "silver_sessions"
]:
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    df = spark.table(f"workspace.default.{name}")
    print("Rows:", df.count())
    df.printSchema()
    display(df.limit(5))



silver_content_catalog
Rows: 2990
root
 |-- available_from_date: string (nullable = true)
 |-- available_to_date: string (nullable = true)
 |-- catalog_status: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- content_type: string (nullable = true)
 |-- creator_label: string (nullable = true)
 |-- duration_seconds: long (nullable = true)
 |-- genre: string (nullable = true)
 |-- is_platform_original: boolean (nullable = true)
 |-- language: string (nullable = true)
 |-- maturity_band: string (nullable = true)
 |-- release_year: long (nullable = true)
 |-- series_collection_id: string (nullable = true)
 |-- title_label: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- silver_processed_time: timestamp (nullable = true)



available_from_date,available_to_date,catalog_status,content_id,content_type,creator_label,duration_seconds,genre,is_platform_original,language,maturity_band,release_year,series_collection_id,title_label,ingestion_timestamp,silver_processed_time
1999-01-01,null,LIMITED,C000006,VIDEO,CreatorGroup_0006,6503,Documentary,false,Hindi,U/A 7+,1999,COL00001,Silver Horizons 0006,2026-08-05T14:15:33.209Z,2026-09-05T06:28:31.144Z
2019-01-01,null,AVAILABLE,C000016,MUSIC,CreatorGroup_0016,240,Devotional,false,English,U/A 13+,2019,null,Midnight Journeys 0016,2026-08-05T14:15:33.209Z,2026-09-05T06:28:31.144Z
2001-01-01,null,LIMITED,C000027,VIDEO,CreatorGroup_0027,2945,Learning,false,English,U/A 7+,2001,COL00004,Open Voices 0027,2026-08-05T14:15:33.209Z,2026-09-05T06:28:31.144Z
2001-01-01,null,AVAILABLE,C000044,PODCAST,CreatorGroup_0044,2667,History,false,Telugu,U,2001,null,Hidden Patterns 0044,2026-08-05T14:15:33.209Z,2026-09-05T06:28:31.144Z
1998-01-01,null,AVAILABLE,C000085,PODCAST,CreatorGroup_0085,1962,Careers,true,Tamil,U/A 16+,1998,null,Parallel Notes 0085,2026-08-05T14:15:33.209Z,2026-09-05T06:28:31.144Z



silver_users
Rows: 24975
root
 |-- user_id: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- signup_cohort: string (nullable = true)
 |-- age_band: string (nullable = true)
 |-- region_band: string (nullable = true)
 |-- preferred_device_segment: string (nullable = true)
 |-- acquisition_channel: string (nullable = true)
 |-- user_status: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- silver_processed_time: timestamp (nullable = true)



user_id,signup_date,signup_cohort,age_band,region_band,preferred_device_segment,acquisition_channel,user_status,ingestion_time,silver_processed_time
U000005,2024-10-26,2024-Q4,55+,North-Urban,TV-first,Campaign,ACTIVE,2026-07-31T09:22:06.553Z,2026-09-05T06:27:24.842Z
U000048,2024-05-28,2024-Q2,25-34,North-Urban,Mobile-first,Organic,ACTIVE,2026-07-31T09:22:06.553Z,2026-09-05T06:27:24.842Z
U000068,2025-01-06,2025-Q1,18-24,East-Urban,Mobile-first,Partner,ACTIVE,2026-07-31T09:22:06.553Z,2026-09-05T06:27:24.842Z
U000072,2025-10-15,2025-Q4,18-24,East-Urban,TV-first,Organic,ACTIVE,2026-07-31T09:22:06.553Z,2026-09-05T06:27:24.842Z
U000076,2024-12-12,2024-Q4,55+,West-Urban,Desktop-first,Organic,ACTIVE,2026-07-31T09:22:06.553Z,2026-09-05T06:27:24.842Z



silver_subscriptions
Rows: 34980
root
 |-- subscription_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- plan_code: string (nullable = true)
 |-- billing_cycle: string (nullable = true)
 |-- period_start_date: date (nullable = true)
 |-- period_end_date: date (nullable = true)
 |-- lifecycle_status: string (nullable = true)
 |-- auto_renew_flag: boolean (nullable = true)
 |-- cancellation_reason_group: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- silver_processed_time: timestamp (nullable = true)



subscription_id,user_id,plan_code,billing_cycle,period_start_date,period_end_date,lifecycle_status,auto_renew_flag,cancellation_reason_group,ingestion_time,silver_processed_time
S0000002,U000002,Premium,Annual,2024-07-06,2026-06-30,ACTIVE,true,null,2026-07-31T09:59:35.869Z,2026-09-05T06:27:58.752Z
S0000022,U000022,Standard,Monthly,2025-01-27,2026-06-30,ACTIVE,true,null,2026-07-31T09:59:35.869Z,2026-09-05T06:27:58.752Z
S0000042,U000042,AudioPlus,Monthly,2024-03-05,2026-06-30,ACTIVE,true,null,2026-07-31T09:59:35.869Z,2026-09-05T06:27:58.752Z
S0000057,U000057,AudioPlus,Monthly,2025-04-04,2026-06-30,ACTIVE,true,null,2026-07-31T09:59:35.869Z,2026-09-05T06:27:58.752Z
S0000061,U000061,Basic,Monthly,2025-07-02,2026-06-30,ACTIVE,true,null,2026-07-31T09:59:35.869Z,2026-09-05T06:27:58.752Z



silver_sessions
Rows: 249701
root
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- subscription_id: string (nullable = true)
 |-- session_start_ts: timestamp_ntz (nullable = true)
 |-- session_end_ts: timestamp_ntz (nullable = true)
 |-- device_type: string (nullable = true)
 |-- consumed_seconds: integer (nullable = true)
 |-- max_playback_position_seconds: integer (nullable = true)
 |-- end_reason: string (nullable = true)
 |-- reported_completed_flag: boolean (nullable = true)
 |-- reported_skipped_flag: boolean (nullable = true)
 |-- source_app_channel: string (nullable = true)
 |-- source_app_version: string (nullable = true)
 |-- producer_batch_id: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- silver_processed_time: timestamp (nullable = true)



session_id,user_id,content_id,subscription_id,session_start_ts,session_end_ts,device_type,consumed_seconds,max_playback_position_seconds,end_reason,reported_completed_flag,reported_skipped_flag,source_app_channel,source_app_version,producer_batch_id,ingestion_timestamp,silver_processed_time
EGS000000003,U018708,C000580,S0018708,2026-02-22T04:54:14.000,2026-02-22T05:50:47.000,Mobile,3368,3368,completed,true,false,tv_app,7.3,BATCH_2026Q1_A,2026-08-05T14:17:54.268Z,2026-09-05T06:28:15.741Z
EGS000000005,U011749,C002597,S0011749,2026-03-11T13:29:32.000,2026-03-11T13:34:00.000,Mobile,104,104,user_stopped,false,false,tv_app,7.3,BATCH_2026Q1_C,2026-08-05T14:17:54.268Z,2026-09-05T06:28:15.741Z
EGS000000016,U005348,C001997,S0005348,2026-01-28T02:23:47.000,2026-01-28T03:37:53.000,Mobile,4309,4309,app_closed,false,false,web_player,6.5,BATCH_2026Q1_B,2026-08-05T14:17:54.268Z,2026-09-05T06:28:15.741Z
EGS000000031,U007892,C002035,S0007892,2026-02-14T23:09:34.000,2026-02-15T00:09:35.000,Mobile,3593,3593,completed,true,false,mobile_app,8.5,BATCH_2026Q1_B,2026-08-05T14:17:54.268Z,2026-09-05T06:28:15.741Z
EGS000000035,U019983,C002685,S0019983,2026-03-12T03:32:47.000,2026-03-12T03:38:00.000,Web,203,203,user_stopped,false,false,tv_app,6.5,BATCH_2026Q1_C,2026-08-05T14:17:54.268Z,2026-09-05T06:28:15.741Z


## 3. Approved DQ rule catalogue

| Rule | Entity | Severity | Destination |
|---|---|---|---|
| DQ-CNT-001 | Content | CRITICAL | quarantine_content |
| DQ-USR-001 | Users | CRITICAL | quarantine_users |
| DQ-SUB-001 | Subscriptions + users | CRITICAL | quarantine_subscriptions |
| DQ-SES-001 | Sessions | CRITICAL | quarantine_sessions |
| DQ-SES-002 | Sessions + trusted references | CRITICAL | quarantine_sessions |
| DQ-SES-003 | Sessions | MAJOR | quarantine_sessions |
| DQ-SES-004 | Sessions + content duration | MAJOR | quarantine_sessions |
| DQ-SES-005 | Sessions | MAJOR | quarantine_sessions |

The conditions below follow the Team 04 Week-06 rulebook. Approved value dictionaries are deliberately not invented here.


In [0]:
# Configuration: field aliases allow the notebook to work with the exact Week-5
# column names without silently renaming fields.
from pyspark.sql import functions as F
from functools import reduce

CATALOG = "workspace.default"

TABLES = {
    "content": f"{CATALOG}.silver_content_catalog",
    "users": f"{CATALOG}.silver_users",
    "subscriptions": f"{CATALOG}.silver_subscriptions",
    "sessions": f"{CATALOG}.silver_sessions",
}

OUTPUTS = {
    "content_trusted": f"{CATALOG}.trusted_content",
    "content_quarantine": f"{CATALOG}.quarantine_content",
    "users_trusted": f"{CATALOG}.trusted_users",
    "users_quarantine": f"{CATALOG}.quarantine_users",
    "subscriptions_trusted": f"{CATALOG}.trusted_subscriptions",
    "subscriptions_quarantine": f"{CATALOG}.quarantine_subscriptions",
    "sessions_trusted": f"{CATALOG}.trusted_sessions",
    "sessions_quarantine": f"{CATALOG}.quarantine_sessions",
}

ALIASES = {
    "content_id": ["content_id"],
    "content_type": ["content_type", "category", "content_category"],
    "content_title": ["title", "content_title"],
    "content_genre": ["genre"],
    "content_language": ["language"],
    "content_duration": ["duration_seconds", "content_duration_seconds", "duration"],
    "release_year": ["release_year"],
    "available_from": ["available_from_date", "available_from"],
    "available_to": ["available_to_date", "available_to"],

    "user_id": ["user_id"],
    "signup_date": ["signup_date"],
    "cohort": ["cohort", "cohort_quarter"],
    "age_band": ["age_band"],
    "region": ["region"],
    "device": ["primary_device", "device"],
    "acquisition": ["acquisition_channel", "acquisition_source"],
    "user_status": ["status", "lifecycle_status"],

    "subscription_id": ["subscription_id"],
    "subscription_user_id": ["user_id", "subscriber_user_id"],
    "period_start": ["period_start_date", "start_date"],
    "period_end": ["period_end_date", "end_date"],
    "plan": ["plan", "plan_name"],
    "billing": ["billing_cycle", "billing_frequency", "billing_type"],
    "subscription_status": ["status", "lifecycle_status"],
    "auto_renew": ["auto_renew_flag"],

    "session_id": ["session_id"],
    "session_user_id": ["user_id"],
    "session_content_id": ["content_id"],
    "source_subscription_id": ["subscription_id", "source_subscription_id"],
    "session_start": ["session_start_ts", "session_start_time", "start_time"],
    "session_end": ["session_end_ts", "session_end_time", "end_time"],
    "consumed_seconds": ["consumed_seconds", "watch_seconds", "duration_seconds"],
    "max_position": ["max_playback_position_seconds", "max_playback_position"],
    "completion_ratio": ["completion_ratio", "completion"],
    "end_reason": ["end_reason"],
    "complete_flag": ["completed_flag", "complete_flag", "completed"],
    "skip_flag": ["skip_flag", "skipped"],
}

def resolve_col(df, logical_name, required=False):
    for candidate in ALIASES.get(logical_name, []):
        if candidate in df.columns:
            return candidate
    if required:
        raise ValueError(
            f"Required field '{logical_name}' not found in {df.columns}. "
            "Check the approved Week-5 schema before continuing."
        )
    return None

def null_or_blank(c):
    return F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == "")

# Approved dictionaries must come from the Team 04 project data dictionary/rulebook.
# Leave a list empty only until the approved dictionary has been copied here.
APPROVED_CONTENT_TYPES = []
APPROVED_CONTENT_CATEGORIES = []
APPROVED_PLANS = []
APPROVED_BILLING = []
APPROVED_SUB_STATUS = []
APPROVED_AGE_BANDS = []
APPROVED_REGIONS = []
APPROVED_DEVICES = []
APPROVED_ACQUISITION = []
APPROVED_USER_STATUS = []

DQ_RUN_ID = "W06_BINGEMETRICS_DQ_RUN_01"


## 4. Candidate physical-record identity and reusable helpers

The DQ layer does **not** use `DISTINCT` to hide duplicates. A stable record key is created from the available business key plus a row fingerprint. This lets reconciliation compare physical Candidate rows without deleting any duplicate records.


In [0]:
from pyspark.sql import Window

def add_record_key(df, business_key):
    # Stable enough for a controlled batch: business key + hash of all original fields.
    cols = [F.col(c).cast("string") for c in df.columns]
    return (
        df.withColumn("_dq_record_hash", F.sha2(F.concat_ws("||", *cols), 256))
          .withColumn("record_key",
                      F.sha2(F.concat_ws("||",
                                         F.coalesce(F.col(business_key).cast("string"), F.lit("")),
                                         F.col("_dq_record_hash")), 256))
    )

def add_common_metadata(df, affected_field_col, reason_col, failed_ids_col, severity_col, status_col):
    return (
        df.withColumn("dq_status", F.col(status_col))
          .withColumn("failed_rule_ids", F.col(failed_ids_col))
          .withColumn("failure_reasons", F.col(reason_col))
          .withColumn("severity", F.col(severity_col))
          .withColumn("affected_field", F.col(affected_field_col))
          .withColumn("dq_run_id", F.lit(DQ_RUN_ID))
          .withColumn("dq_checked_ts", F.current_timestamp())
          .withColumn("source_file",
                      F.coalesce(F.col("_source_file_name"), F.lit(None).cast("string")))
          .withColumn("batch_id",
                      F.coalesce(F.col("_ingestion_run_id"), F.lit(None).cast("string")))
          .withColumn("quarantined_at",
                      F.when(F.col(status_col) == "FAIL", F.current_timestamp()))
          .withColumn("rework_status", F.lit("ORIGINAL_DQ"))
    )

def rule_array(rule_exprs):
    return F.array(*[
        F.when(expr, F.lit(rule_id)).otherwise(F.lit(None))
        for rule_id, expr in rule_exprs
    ])

def compact_array(arr_col):
    return F.expr(f"filter({arr_col}, x -> x is not null)")

def reasons_array(rule_exprs):
    return F.array(*[
        F.when(expr, F.lit(reason)).otherwise(F.lit(None))
        for _, expr, reason in rule_exprs
    ])

def severity_expr(failed_ids):
    return (
        F.when(F.array_contains(failed_ids, "DQ-SES-001"), "CRITICAL")
         .when(F.array_contains(failed_ids, "DQ-SES-002"), "CRITICAL")
         .when(F.array_contains(failed_ids, "DQ-CNT-001"), "CRITICAL")
         .when(F.array_contains(failed_ids, "DQ-SUB-001"), "CRITICAL")
         .when(F.array_contains(failed_ids, "DQ-USR-001"), "CRITICAL")
         .when(F.size(failed_ids) > 0, "MAJOR")
         .otherwise("NONE")
    )


## 5. Content DQ — DQ-CNT-001

Fails when the content ID is missing/duplicated, required content fields are missing, approved type/category values are violated, duration/release year is invalid, or availability dates are invalid.


In [0]:
content = spark.table(TABLES["content"])

content_id = resolve_col(content, "content_id", True)
content_duration = resolve_col(content, "content_duration")
release_year = resolve_col(content, "release_year")
available_from = resolve_col(content, "available_from")
available_to = resolve_col(content, "available_to")
content_type = resolve_col(content, "content_type")
content_title = resolve_col(content, "content_title")

# Create stable record key BEFORE adding temporary columns
content = add_record_key(content, content_id)

# Count duplicate content IDs
w = Window.partitionBy(F.col(content_id))

content = content.withColumn(
    "_dup_count",
    F.count("*").over(w)
)

# -----------------------------
# DQ-CNT-001 checks
# -----------------------------

content_fail = (
    null_or_blank(content_id) |
    (F.col("_dup_count") > 1)
)

# Required title
if content_title:
    content_fail = content_fail | null_or_blank(content_title)

# Duration must be positive
if content_duration:
    content_fail = content_fail | (
        F.col(content_duration).cast("double").isNull() |
        (F.col(content_duration).cast("double") <= 0)
    )

# Release year must be positive
if release_year:
    content_fail = content_fail | (
        F.col(release_year).cast("int").isNull() |
        (F.col(release_year).cast("int") <= 0)
    )

# Available-to date cannot be before available-from date
if available_from and available_to:
    content_fail = content_fail | (
        F.to_date(F.col(available_to)) <
        F.to_date(F.col(available_from))
    )

# Approved content type
if content_type and APPROVED_CONTENT_TYPES:
    content_fail = content_fail | (
        (~F.col(content_type).isin(APPROVED_CONTENT_TYPES)) &
        F.col(content_type).isNotNull()
    )

# -----------------------------
# Failure information
# -----------------------------

empty_string_array = F.array().cast("array<string>")

failed = F.when(
    content_fail,
    F.array(F.lit("DQ-CNT-001"))
).otherwise(
    empty_string_array
)

reasons = F.when(
    content_fail,
    F.array(
        F.lit(
            "Catalog integrity rule failed: missing/duplicate key, "
            "required field, approved value, duration/year, "
            "or availability condition."
        )
    )
).otherwise(
    empty_string_array
)

# -----------------------------
# Create checked table
# -----------------------------

content_checked = (
    content
    .withColumn("failed_rule_ids", failed)
    .withColumn("failure_reasons", reasons)
    .withColumn(
        "severity",
        F.when(content_fail, F.lit("CRITICAL"))
         .otherwise(F.lit("NONE"))
    )
    .withColumn(
        "affected_field",
        F.when(
            content_fail,
            F.lit("content_id / required catalog fields")
        ).otherwise(
            F.lit(None).cast("string")
        )
    )
    .withColumn(
        "dq_status",
        F.when(content_fail, F.lit("FAIL"))
         .otherwise(F.lit("PASS"))
    )
    .withColumn("dq_run_id", F.lit(DQ_RUN_ID))
    .withColumn("dq_checked_ts", F.current_timestamp())
    .withColumn(
        "source_file",
        F.col("_source_file_name")
        if "_source_file_name" in content.columns
        else F.lit(None).cast("string")
    )
    .withColumn(
        "batch_id",
        F.col("_ingestion_run_id")
        if "_ingestion_run_id" in content.columns
        else F.lit(None).cast("string")
    )
    .withColumn(
        "quarantined_at",
        F.when(
            content_fail,
            F.current_timestamp()
        )
    )
    .withColumn("rework_status", F.lit("ORIGINAL_DQ"))
    .drop("_dup_count", "_dq_record_hash")
)

# -----------------------------
# Results
# -----------------------------

content_checked.groupBy("dq_status").count().show()

display(
    content_checked.select(
        "record_key",
        "dq_status",
        "failed_rule_ids",
        "failure_reasons",
        "severity",
        "affected_field"
    ).limit(20)
)

+---------+-----+
|dq_status|count|
+---------+-----+
|     PASS| 2980|
|     FAIL|   10|
+---------+-----+



record_key,dq_status,failed_rule_ids,failure_reasons,severity,affected_field
2efb13832e5147cdfd0eb80aeca588c02922c114ec7e559fd70cdca9e1cb64b8,PASS,List(),List(),NONE,null
d16a63cd906e50045310afef2c702d0a0f8d366da27913f0757caf2b11caa91f,PASS,List(),List(),NONE,null
20744a52e3b13c036621463f23931fbad4b4872852e33f3021536afdca83c1cb,PASS,List(),List(),NONE,null
6b296a483eca76e98f18ee89b8a272f5e64c98566058e8d231eca0ae317f7fd7,PASS,List(),List(),NONE,null
1b5c6288eb6e8a073c7be6314d0c85d470578d2b12863097cddd00e204daad0e,PASS,List(),List(),NONE,null
1587b70fd41f15025bf262d226820de0cc518deae9dad0f94c1d852036620bbe,PASS,List(),List(),NONE,null
03d4f1eac3a97da94dcab195342a7415d887ed65fed47f086cf74c131ca99bc3,PASS,List(),List(),NONE,null
abeb22f0135c0b6283f978c52b120b1cd5763e765d369873d4fd386bd6bcf10f,PASS,List(),List(),NONE,null
48941192f3a682cd600fa2810e3894b29f154a81a0e483ad91fa73ed5eafce95,PASS,List(),List(),NONE,null
552edcdcfb5e4b709853e41528afd4b61b38303f81e0f9c96456430538a461c5,PASS,List(),List(),NONE,null


In [0]:
content_trusted = content_checked.filter("dq_status = 'PASS'")
content_quarantine = content_checked.filter("dq_status = 'FAIL'")

content_trusted.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(OUTPUTS["content_trusted"])
content_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(OUTPUTS["content_quarantine"])


## 6. Users DQ — DQ-USR-001

Fails when `user_id` is missing/duplicated, signup/cohort is invalid or inconsistent, an approved coarse category is violated, or prohibited identifying detail is present.

The approved category values must come from the project data dictionary; this notebook does not invent them.


In [0]:
users = spark.table(TABLES["users"])

user_id = resolve_col(users, "user_id", True)
signup_date = resolve_col(users, "signup_date")
cohort = resolve_col(users, "cohort")
age_band = resolve_col(users, "age_band")
region = resolve_col(users, "region")
device = resolve_col(users, "device")
acquisition = resolve_col(users, "acquisition")
user_status = resolve_col(users, "user_status")

# Create record key before temporary columns
users = add_record_key(users, user_id)

# Count duplicate user IDs
w = Window.partitionBy(F.col(user_id))

users = users.withColumn(
    "_dup_count",
    F.count("*").over(w)
)

# -----------------------------
# DQ-USR-001 checks
# -----------------------------

user_fail = (
    null_or_blank(user_id) |
    (F.col("_dup_count") > 1)
)

# Signup date must exist
if signup_date:
    user_fail = user_fail | F.col(signup_date).isNull()

# -----------------------------
# Cohort validation
# -----------------------------

if signup_date and cohort:

    cohort_year = F.regexp_extract(
        F.col(cohort).cast("string"),
        r"^(\d{4})-Q([1-4])$",
        1
    ).cast("int")

    cohort_quarter = F.regexp_extract(
        F.col(cohort).cast("string"),
        r"^(\d{4})-Q([1-4])$",
        2
    ).cast("int")

    signup_year = F.year(
        F.to_date(F.col(signup_date))
    )

    signup_quarter = F.quarter(
        F.to_date(F.col(signup_date))
    )

    valid_cohort_format = (
        F.regexp_extract(
            F.col(cohort).cast("string"),
            r"^(\d{4})-Q([1-4])$",
            0
        ) != ""
    )

    # Signup date cannot be after the cohort period
    cohort_after_signup = (
        valid_cohort_format &
        (
            (signup_year > cohort_year) |
            (
                (signup_year == cohort_year) &
                (signup_quarter > cohort_quarter)
            )
        )
    )

    user_fail = user_fail | cohort_after_signup


# -----------------------------
# Approved category checks
# -----------------------------

def add_allowed_failure(current, col_name, values):
    if col_name and values:
        return current | (
            (~F.col(col_name).isin(values)) &
            F.col(col_name).isNotNull()
        )
    return current


user_fail = add_allowed_failure(
    user_fail,
    age_band,
    APPROVED_AGE_BANDS
)

user_fail = add_allowed_failure(
    user_fail,
    region,
    APPROVED_REGIONS
)

user_fail = add_allowed_failure(
    user_fail,
    device,
    APPROVED_DEVICES
)

user_fail = add_allowed_failure(
    user_fail,
    acquisition,
    APPROVED_ACQUISITION
)

user_fail = add_allowed_failure(
    user_fail,
    user_status,
    APPROVED_USER_STATUS
)


# -----------------------------
# Prohibited identifying details
# -----------------------------

PROHIBITED_IDENTIFYING_PATTERNS = []

if PROHIBITED_IDENTIFYING_PATTERNS:

    check_columns = [
        c for c in users.columns
        if not c.startswith("_")
        and c not in ["record_key"]
    ]

    if check_columns:
        combined = F.concat_ws(
            "||",
            *[
                F.col(c).cast("string")
                for c in check_columns
            ]
        )

        for pattern in PROHIBITED_IDENTIFYING_PATTERNS:
            user_fail = user_fail | combined.rlike(pattern)


# -----------------------------
# Failure information
# -----------------------------

empty_string_array = F.array().cast("array<string>")

failed = F.when(
    user_fail,
    F.array(F.lit("DQ-USR-001"))
).otherwise(
    empty_string_array
)

reasons = F.when(
    user_fail,
    F.array(
        F.lit(
            "Pseudonymous-user boundary rule failed: "
            "missing/duplicate user key, signup/cohort condition, "
            "approved coarse category, or identifying-detail condition."
        )
    )
).otherwise(
    empty_string_array
)


# -----------------------------
# Create checked table
# -----------------------------

users_checked = (
    users
    .withColumn("failed_rule_ids", failed)
    .withColumn("failure_reasons", reasons)
    .withColumn(
        "severity",
        F.when(user_fail, F.lit("CRITICAL"))
         .otherwise(F.lit("NONE"))
    )
    .withColumn(
        "affected_field",
        F.when(
            user_fail,
            F.lit(
                "user_id / signup_date / cohort / "
                "approved coarse categories"
            )
        ).otherwise(
            F.lit(None).cast("string")
        )
    )
    .withColumn(
        "dq_status",
        F.when(user_fail, F.lit("FAIL"))
         .otherwise(F.lit("PASS"))
    )
    .withColumn("dq_run_id", F.lit(DQ_RUN_ID))
    .withColumn("dq_checked_ts", F.current_timestamp())
    .withColumn(
        "source_file",
        F.col("_source_file_name")
        if "_source_file_name" in users.columns
        else F.lit(None).cast("string")
    )
    .withColumn(
        "batch_id",
        F.col("_ingestion_run_id")
        if "_ingestion_run_id" in users.columns
        else F.lit(None).cast("string")
    )
    .withColumn(
        "quarantined_at",
        F.when(
            user_fail,
            F.current_timestamp()
        )
    )
    .withColumn(
        "rework_status",
        F.lit("ORIGINAL_DQ")
    )
    .drop("_dup_count", "_dq_record_hash")
)


# -----------------------------
# Results
# -----------------------------

users_checked.groupBy("dq_status").count().show()

display(
    users_checked.select(
        "record_key",
        "user_id",
        "dq_status",
        "failed_rule_ids",
        "failure_reasons",
        "severity",
        "affected_field"
    ).limit(20)
)

+---------+-----+
|dq_status|count|
+---------+-----+
|     PASS|24975|
+---------+-----+



record_key,user_id,dq_status,failed_rule_ids,failure_reasons,severity,affected_field
155c7401c271f809f2f47fa49d21f1107556caaa3438379f5a9f4e7c53fd1dc9,U000001,PASS,List(),List(),NONE,null
1f2f0b30f5291bab2a71daa267bfbf5af7315f7d5633acf19b47b8b4eed96aa1,U000002,PASS,List(),List(),NONE,null
b850a36c448cb9709d72c98493519b2eec5c91b874979d6027c50241cabd4b58,U000003,PASS,List(),List(),NONE,null
6eee0a28eb8f57ba25b274e006a3a13f56d75e9b4e8eebc37814898b611fab1e,U000004,PASS,List(),List(),NONE,null
ee155f68ebacbe299c9c3f0e806cd2496e16a938e8eedf2557d49f90294bddcf,U000005,PASS,List(),List(),NONE,null
e8a94af69430a43293758ea4d783f7d657b6921243a1a3f4f1ba3ca87cbbde67,U000006,PASS,List(),List(),NONE,null
e195e284a3bf80bb6c7776c2edec9a576d2c186e686bb6ca5253258f6cf70699,U000007,PASS,List(),List(),NONE,null
c3beb0af45ea5d1e2289fb44e732ae0d8a98e1929c986081ed20981986cd476d,U000008,PASS,List(),List(),NONE,null
ab3e87d9e94dfe387915ddc63218a5866b7b20ed2c323a973ae686658de954b4,U000009,PASS,List(),List(),NONE,null
c5ca697afbf1bbc84c29eae9510ba5dd361cd38c18656ec379c7ebccb4f56c56,U000010,PASS,List(),List(),NONE,null


In [0]:
users_trusted = users_checked.filter("dq_status = 'PASS'")
users_quarantine = users_checked.filter("dq_status = 'FAIL'")

users_trusted.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(OUTPUTS["users_trusted"])
users_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(OUTPUTS["users_quarantine"])


## 7. Subscriptions DQ — DQ-SUB-001

Fails when subscription ID is missing/duplicated, user is not found, period is invalid, approved plan/billing/status values are violated, status/dates are inconsistent, or incompatible periods overlap for the same user.


In [0]:
subs = spark.table(TABLES["subscriptions"])

subscription_id = resolve_col(subs, "subscription_id", True)
subscription_user_id = resolve_col(subs, "subscription_user_id", True)
period_start = resolve_col(subs, "period_start", True)
period_end = resolve_col(subs, "period_end", True)
plan = resolve_col(subs, "plan")
billing = resolve_col(subs, "billing")
subscription_status = resolve_col(subs, "subscription_status")

# --------------------------------------------------
# Create record key
# --------------------------------------------------

subs = add_record_key(subs, subscription_id)

# --------------------------------------------------
# Check whether subscription user exists in Trusted Users
# --------------------------------------------------

trusted_user_ids = (
    users_trusted
    .select(F.col(user_id).alias("_trusted_user_id"))
    .dropDuplicates()
)

subs = (
    subs
    .join(
        trusted_user_ids,
        F.col(subscription_user_id) == F.col("_trusted_user_id"),
        "left"
    )
    .withColumn(
        "_user_resolves",
        F.col("_trusted_user_id").isNotNull()
    )
    .drop("_trusted_user_id")
)

# --------------------------------------------------
# Duplicate subscription IDs
# --------------------------------------------------

w_sub = Window.partitionBy(F.col(subscription_id))

subs = subs.withColumn(
    "_dup_count",
    F.count("*").over(w_sub)
)

# --------------------------------------------------
# Check overlapping subscription periods
# --------------------------------------------------

overlap = (
    subs.alias("a")
    .join(
        subs.alias("b"),
        (
            F.col(f"a.{subscription_user_id}") ==
            F.col(f"b.{subscription_user_id}")
        )
        &
        (
            F.col(f"a.{subscription_id}") !=
            F.col(f"b.{subscription_id}")
        )
        &
        (
            F.to_date(F.col(f"a.{period_start}")) <=
            F.to_date(F.col(f"b.{period_end}"))
        )
        &
        (
            F.to_date(F.col(f"b.{period_start}")) <=
            F.to_date(F.col(f"a.{period_end}"))
        ),
        "left_semi"
    )
    .select(
        F.col(f"a.{subscription_id}")
        .alias("_overlap_subscription_id")
    )
    .dropDuplicates()
)

subs = (
    subs
    .join(
        overlap,
        F.col(subscription_id) ==
        F.col("_overlap_subscription_id"),
        "left"
    )
    .withColumn(
        "_overlap",
        F.col("_overlap_subscription_id").isNotNull()
    )
    .drop("_overlap_subscription_id")
)

# --------------------------------------------------
# DQ-SUB-001 checks
# --------------------------------------------------

sub_fail = (
    null_or_blank(subscription_id)
    |
    (F.col("_dup_count") > 1)
    |
    (~F.col("_user_resolves"))
    |
    F.col(period_start).isNull()
    |
    F.col(period_end).isNull()
    |
    (
        F.to_date(F.col(period_start)) >
        F.to_date(F.col(period_end))
    )
)

# Approved plan values
if plan and APPROVED_PLANS:
    sub_fail = sub_fail | (
        (~F.col(plan).isin(APPROVED_PLANS))
        & F.col(plan).isNotNull()
    )

# Approved billing values
if billing and APPROVED_BILLING:
    sub_fail = sub_fail | (
        (~F.col(billing).isin(APPROVED_BILLING))
        & F.col(billing).isNotNull()
    )

# Approved subscription status values
if subscription_status and APPROVED_SUB_STATUS:
    sub_fail = sub_fail | (
        (~F.col(subscription_status).isin(APPROVED_SUB_STATUS))
        & F.col(subscription_status).isNotNull()
    )

# Overlapping subscriptions
sub_fail = sub_fail | F.col("_overlap")

# --------------------------------------------------
# Failure information
# --------------------------------------------------

empty_string_array = F.array().cast("array<string>")

failed = F.when(
    sub_fail,
    F.array(F.lit("DQ-SUB-001"))
).otherwise(
    empty_string_array
)

reasons = F.when(
    sub_fail,
    F.array(
        F.lit(
            "Subscription validity rule failed: "
            "missing/duplicate key, invalid user reference, "
            "invalid period, approved value, status/date condition, "
            "or incompatible period overlap."
        )
    )
).otherwise(
    empty_string_array
)

# --------------------------------------------------
# Create checked table
# --------------------------------------------------

subs_checked = (
    subs
    .withColumn("failed_rule_ids", failed)
    .withColumn("failure_reasons", reasons)
    .withColumn(
        "severity",
        F.when(sub_fail, F.lit("CRITICAL"))
         .otherwise(F.lit("NONE"))
    )
    .withColumn(
        "affected_field",
        F.when(
            sub_fail,
            F.lit(
                "subscription_id / user_id / period / "
                "plan / billing / status"
            )
        ).otherwise(
            F.lit(None).cast("string")
        )
    )
    .withColumn(
        "dq_status",
        F.when(sub_fail, F.lit("FAIL"))
         .otherwise(F.lit("PASS"))
    )
    .withColumn("dq_run_id", F.lit(DQ_RUN_ID))
    .withColumn("dq_checked_ts", F.current_timestamp())
    .withColumn(
        "source_file",
        F.col("_source_file_name")
        if "_source_file_name" in subs.columns
        else F.lit(None).cast("string")
    )
    .withColumn(
        "batch_id",
        F.col("_ingestion_run_id")
        if "_ingestion_run_id" in subs.columns
        else F.lit(None).cast("string")
    )
    .withColumn(
        "quarantined_at",
        F.when(
            sub_fail,
            F.current_timestamp()
        )
    )
    .withColumn(
        "rework_status",
        F.lit("ORIGINAL_DQ")
    )
    .drop(
        "_dup_count",
        "_user_resolves",
        "_overlap",
        "_dq_record_hash"
    )
)

# --------------------------------------------------
# Results
# --------------------------------------------------

subs_checked.groupBy("dq_status").count().show()

display(
    subs_checked.select(
        "record_key",
        subscription_id,
        "dq_status",
        "failed_rule_ids",
        "failure_reasons",
        "severity",
        "affected_field"
    ).limit(20)
)

+---------+-----+
|dq_status|count|
+---------+-----+
|     PASS|34755|
|     FAIL|  225|
+---------+-----+



record_key,subscription_id,dq_status,failed_rule_ids,failure_reasons,severity,affected_field
bb8db683093556a4785585ba996f235e647f65e37e49cfd2fea7cfc9d0a2462b,S0000002,PASS,List(),List(),NONE,null
085279b2cfe34912a24d0455be802404f095aab7f5bd915971287486cac736b2,S0000005,PASS,List(),List(),NONE,null
4003b75b2d2f4068cbca9834e7f30a2b65babcb0b4bd909346d003b22eb37436,S0000009,PASS,List(),List(),NONE,null
6f3a46f680c93d0a86b369d1056ffef8606df2e52f4b3acec8dd7c5ed6a1610e,S0000012,PASS,List(),List(),NONE,null
82fea3cdedcc3f5c4f461cb5060293a06f4777c60da1b8ab373a376cb16eada7,S0000013,PASS,List(),List(),NONE,null
4a54a01c0d0f98b6287c059908b9455bb74ec5ad7ef1cfa6d3d82a776983e429,S0000017,PASS,List(),List(),NONE,null
14ce37eea48cfa471fe76222e4c9edd56acef8f4d57fb15d3cffc5f24413d740,S0000022,PASS,List(),List(),NONE,null
244e6312b1cd57a41c3bcfb11d1ba9fce198969ddb48b1b98e770e044f8eb609,S0000023,PASS,List(),List(),NONE,null
caf115ed7a3dbda1157673cc8255801f483bb008070d1a4b9795b685fc7fae6d,S0000025,PASS,List(),List(),NONE,null
c863d592279efe20e703f637216eb54d3477dce9c150d3e044d2742c9931dbb7,S0000028,PASS,List(),List(),NONE,null


In [0]:
subs_trusted = subs_checked.filter("dq_status = 'PASS'")
subs_quarantine = subs_checked.filter("dq_status = 'FAIL'")

subs_trusted.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(OUTPUTS["subscriptions_trusted"])
subs_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(OUTPUTS["subscriptions_quarantine"])


## 8. Sessions DQ — DQ-SES-001 to DQ-SES-005

Sessions are evaluated **after** Trusted Users, Trusted Content and Trusted Subscriptions exist.

The subscription check is performed against Trusted Subscriptions and requires exactly one qualifying subscription for the same user, covering the session start and matching the source subscription ID.


In [0]:
# ============================================================
# SESSIONS DQ
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

sessions = spark.table(TABLES["sessions"])

# ------------------------------------------------------------
# Resolve columns
# ------------------------------------------------------------

session_id = resolve_col(sessions, "session_id", True)
session_user_id = resolve_col(sessions, "session_user_id", True)
session_content_id = resolve_col(sessions, "session_content_id", True)
source_subscription_id = resolve_col(sessions, "source_subscription_id")
session_start = resolve_col(sessions, "session_start", True)
session_end = resolve_col(sessions, "session_end", True)
consumed_seconds = resolve_col(sessions, "consumed_seconds")
max_position = resolve_col(sessions, "max_position")
completion_ratio = resolve_col(sessions, "completion_ratio")
end_reason = resolve_col(sessions, "end_reason")
complete_flag = resolve_col(sessions, "complete_flag")
skip_flag = resolve_col(sessions, "skip_flag")

# ------------------------------------------------------------
# Record key
# ------------------------------------------------------------

sessions = add_record_key(sessions, session_id)

# ------------------------------------------------------------
# DQ-SES-001
# ------------------------------------------------------------

w = Window.partitionBy(F.col(session_id))

sessions = sessions.withColumn(
    "_dup_count",
    F.count("*").over(w)
)

r001 = (
    null_or_blank(session_id)
    | (F.col("_dup_count") > 1)
)

# ------------------------------------------------------------
# DQ-SES-002
# User reference
# ------------------------------------------------------------

trusted_users = (
    users_trusted
    .select(
        F.col(user_id).cast("string").alias("_user_ref")
    )
    .dropDuplicates()
)

sessions = sessions.withColumn(
    "_session_user_ref",
    F.col(session_user_id).cast("string")
)

sessions = (
    sessions
    .join(
        trusted_users,
        F.col("_session_user_ref") == F.col("_user_ref"),
        "left"
    )
    .withColumn(
        "_user_ok",
        F.col("_user_ref").isNotNull()
    )
    .drop("_session_user_ref", "_user_ref")
)

# ------------------------------------------------------------
# Content reference
# ------------------------------------------------------------

trusted_content = (
    content_trusted
    .select(
        F.col(content_id).cast("string").alias("_content_ref")
    )
    .dropDuplicates()
)

sessions = sessions.withColumn(
    "_session_content_ref",
    F.col(session_content_id).cast("string")
)

sessions = (
    sessions
    .join(
        trusted_content,
        F.col("_session_content_ref") == F.col("_content_ref"),
        "left"
    )
    .withColumn(
        "_content_ok",
        F.col("_content_ref").isNotNull()
    )
    .drop("_session_content_ref", "_content_ref")
)

# ------------------------------------------------------------
# Subscription reference
# ------------------------------------------------------------

trusted_subs = (
    subs_trusted
    .select(
        F.col(subscription_id).cast("string").alias("_sub_id"),
        F.col(subscription_user_id).cast("string").alias("_sub_user"),
        F.to_date(F.col(period_start)).alias("_sub_start"),
        F.to_date(F.col(period_end)).alias("_sub_end")
    )
    .dropDuplicates()
)

sessions = sessions.withColumn(
    "_sess_user",
    F.col(session_user_id).cast("string")
)

sessions = sessions.withColumn(
    "_sess_start",
    F.to_timestamp(F.col(session_start))
)

if source_subscription_id:
    sessions = sessions.withColumn(
        "_sess_source_sub",
        F.col(source_subscription_id).cast("string")
    )
else:
    sessions = sessions.withColumn(
        "_sess_source_sub",
        F.lit(None).cast("string")
    )

# ------------------------------------------------------------
# Find covering subscriptions
# ------------------------------------------------------------

sub_check = (
    sessions
    .select(
        "record_key",
        "_sess_user",
        "_sess_start",
        "_sess_source_sub"
    )
    .join(
        trusted_subs,
        (
            F.col("_sess_user") == F.col("_sub_user")
        )
        &
        (
            F.to_date(F.col("_sess_start")) >= F.col("_sub_start")
        )
        &
        (
            F.to_date(F.col("_sess_start")) <= F.col("_sub_end")
        ),
        "left"
    )
    .withColumn(
        "_source_match",
        (
            F.col("_sub_id").isNotNull()
            &
            (F.col("_sess_source_sub") == F.col("_sub_id"))
        )
    )
)

sub_check = (
    sub_check
    .groupBy("record_key")
    .agg(
        F.count("_sub_id").alias("_sub_count"),
        F.max(
            F.when(
                F.col("_source_match"),
                1
            ).otherwise(0)
        ).alias("_source_match")
    )
)

sessions = (
    sessions
    .join(
        sub_check,
        "record_key",
        "left"
    )
)

r002 = (
    (~F.col("_user_ok"))
    |
    (~F.col("_content_ok"))
    |
    (F.coalesce(F.col("_sub_count"), F.lit(0)) != 1)
    |
    (F.coalesce(F.col("_source_match"), F.lit(0)) != 1)
)

# ------------------------------------------------------------
# DQ-SES-003
# Time validity
# ------------------------------------------------------------

start_ts = F.to_timestamp(F.col(session_start))
end_ts = F.to_timestamp(F.col(session_end))

r003 = (
    start_ts.isNull()
    |
    end_ts.isNull()
    |
    (start_ts >= end_ts)
    |
    (
        start_ts <
        F.to_timestamp(F.lit("2026-01-01 00:00:00"))
    )
    |
    (
        end_ts >
        F.to_timestamp(F.lit("2026-03-31 23:59:59"))
    )
)

# ------------------------------------------------------------
# DQ-SES-004
# Playback validity
# ------------------------------------------------------------

r004 = F.lit(False)

if consumed_seconds:
    consumed = F.col(consumed_seconds).cast("double")

    r004 = (
        r004
        | consumed.isNull()
        | (consumed < 0)
    )

if max_position:
    position = F.col(max_position).cast("double")

    r004 = (
        r004
        | position.isNull()
        | (position < 0)
    )

# ------------------------------------------------------------
# Content duration
# ------------------------------------------------------------

duration_col = resolve_col(
    content_trusted,
    "content_duration"
)

if duration_col:

    duration_df = (
        content_trusted
        .select(
            F.col(content_id).cast("string").alias("_dur_content"),
            F.col(duration_col).cast("double").alias("_duration")
        )
        .dropDuplicates(["_dur_content"])
    )

    sessions = sessions.withColumn(
        "_session_content",
        F.col(session_content_id).cast("string")
    )

    sessions = (
        sessions
        .join(
            duration_df,
            F.col("_session_content") == F.col("_dur_content"),
            "left"
        )
        .drop("_session_content", "_dur_content")
    )

    r004 = (
        r004
        |
        F.col("_duration").isNull()
        |
        (F.col("_duration") <= 0)
    )

    if consumed_seconds:
        r004 = (
            r004
            |
            (
                F.col(consumed_seconds).cast("double")
                > F.col("_duration")
            )
        )

    if max_position:
        r004 = (
            r004
            |
            (
                F.col(max_position).cast("double")
                > F.col("_duration")
            )
        )

# ------------------------------------------------------------
# DQ-SES-005
# Outcome consistency
# ------------------------------------------------------------

r005 = F.lit(False)

if completion_ratio:

    ratio = F.col(completion_ratio).cast("double")

    r005 = (
        r005
        | ratio.isNull()
        | (ratio < 0)
        | (ratio > 1)
    )

    if complete_flag:

        complete = F.col(complete_flag).cast("boolean")

        r005 = (
            r005
            |
            (
                ((ratio >= 0.90) != complete)
            )
        )

    if skip_flag:

        skip = F.col(skip_flag).cast("boolean")

        r005 = (
            r005
            |
            (
                ((ratio <= 0.20) != skip)
            )
        )

    if end_reason:

        completed = (
            F.lower(
                F.trim(
                    F.col(end_reason).cast("string")
                )
            )
            == "completed"
        )

        r005 = (
            r005
            |
            (
                completed
                &
                (ratio < 0.90)
            )
        )

if complete_flag and skip_flag:

    r005 = (
        r005
        |
        (
            F.col(complete_flag).cast("boolean")
            &
            F.col(skip_flag).cast("boolean")
        )
    )

# ------------------------------------------------------------
# Create failed rule arrays
# ------------------------------------------------------------

failed_rule_ids = F.array_remove(
    F.array(
        F.when(r001, F.lit("DQ-SES-001")),
        F.when(r002, F.lit("DQ-SES-002")),
        F.when(r003, F.lit("DQ-SES-003")),
        F.when(r004, F.lit("DQ-SES-004")),
        F.when(r005, F.lit("DQ-SES-005"))
    ),
    F.lit(None)
)

failure_reasons = F.array_remove(
    F.array(
        F.when(
            r001,
            F.lit("Session key is missing, blank, or duplicated.")
        ),
        F.when(
            r002,
            F.lit("User, content, or subscription reference check failed.")
        ),
        F.when(
            r003,
            F.lit("Session start/end time is invalid.")
        ),
        F.when(
            r004,
            F.lit("Playback or duration check failed.")
        ),
        F.when(
            r005,
            F.lit("Session outcome consistency check failed.")
        )
    ),
    F.lit(None)
)

# ------------------------------------------------------------
# Final DQ result
# ------------------------------------------------------------

sessions_checked = (
    sessions

    .withColumn(
        "failed_rule_ids",
        failed_rule_ids
    )

    .withColumn(
        "failure_reasons",
        failure_reasons
    )

    .withColumn(
        "dq_status",
        F.when(
            F.size(F.col("failed_rule_ids")) > 0,
            "FAIL"
        ).otherwise("PASS")
    )

    .withColumn(
        "severity",
        F.when(
            F.size(F.col("failed_rule_ids")) == 0,
            "NONE"
        )
        .when(
            F.array_contains(
                F.col("failed_rule_ids"),
                "DQ-SES-001"
            )
            |
            F.array_contains(
                F.col("failed_rule_ids"),
                "DQ-SES-002"
            ),
            "CRITICAL"
        )
        .otherwise("MAJOR")
    )

    .withColumn(
        "affected_field",
        F.when(
            F.size(F.col("failed_rule_ids")) > 0,
            "session_id / references / timestamps / playback / outcome"
        ).otherwise(None)
    )

    .withColumn(
        "dq_run_id",
        F.lit(DQ_RUN_ID)
    )

    .withColumn(
        "dq_checked_ts",
        F.current_timestamp()
    )

    .withColumn(
        "source_file",
        F.col("_source_file_name")
        if "_source_file_name" in sessions.columns
        else F.lit(None).cast("string")
    )

    .withColumn(
        "batch_id",
        F.col("_ingestion_run_id")
        if "_ingestion_run_id" in sessions.columns
        else F.lit(None).cast("string")
    )

    .withColumn(
        "quarantined_at",
        F.when(
            F.size(F.col("failed_rule_ids")) > 0,
            F.current_timestamp()
        ).otherwise(None)
    )

    .withColumn(
        "rework_status",
        F.lit("ORIGINAL_DQ")
    )

    .drop(
        "_dup_count",
        "_user_ok",
        "_content_ok",
        "_sess_user",
        "_sess_start",
        "_sess_source_sub",
        "_sub_count",
        "_source_match",
        "_duration",
        "_dq_record_hash"
    )
)

# ------------------------------------------------------------
# Test
# ------------------------------------------------------------

sessions_checked.groupBy("dq_status").count().show()

display(
    sessions_checked.select(
        "record_key",
        session_id,
        "dq_status",
        "failed_rule_ids",
        "failure_reasons",
        "severity",
        "affected_field"
    ).limit(30)
)

+---------+------+
|dq_status| count|
+---------+------+
|     PASS|249701|
+---------+------+



record_key,session_id,dq_status,failed_rule_ids,failure_reasons,severity,affected_field
52baaef1ea1f3cd3c84677791ccdbf9bdf68afc7229cc028dbc4c299d6bc1e1a,EGS000000003,PASS,null,null,MAJOR,null
fba92daa5e9d3db8ac9bfa5c00c6ac88703f4f98d90609d8531f88c536edf2d7,EGS000000005,PASS,null,null,MAJOR,null
8d4536f6eeb64976c24b88c37717ea89ac3a935bb97977df569f93bb9fb4f83c,EGS000000016,PASS,null,null,MAJOR,null
39ae8c514bf0231ee5a204bcdd8af5e98cffcdd3ff3a7a97c886c0f3ed3aaf30,EGS000000031,PASS,null,null,MAJOR,null
e655ce6a310ad64c9fac4fab6b67921860657e4db1db96e70efb9e4d0cf634b7,EGS000000035,PASS,null,null,MAJOR,null
6a7c3c376ffa8295bcaad0a145e74501d059f474adc3e2b0f73b70e2e40f126f,EGS000000046,PASS,null,null,MAJOR,null
7db3191641ebaa1b43cc65b4d06aca50e3446652368fe17c5e691f0365067044,EGS000000052,PASS,null,null,MAJOR,null
02ad7ee1fdc04f38c8365d244d811316948c6cdc2c745833ff1727e160ab3abc,EGS000000080,PASS,null,null,MAJOR,null
47751a9df65b5f237d6b1105e8113e88066f64bf547e0d9b75676b203b641893,EGS000000098,PASS,null,null,MAJOR,null
d63a1faa067029129474c69b42d5eab59867ea6be0695e899047ae66b297d49a,EGS000000103,PASS,null,null,MAJOR,null


In [0]:
sessions_trusted = sessions_checked.filter("dq_status = 'PASS'")
sessions_quarantine = sessions_checked.filter("dq_status = 'FAIL'")

sessions_trusted.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(OUTPUTS["sessions_trusted"])
sessions_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(OUTPUTS["sessions_quarantine"])


## 9. Rule-failure scorecard

This is the evidence view for the eight approved rules. A zero count is valid evidence.


In [0]:
from functools import reduce

# ============================================================
# DQ RULE SCORECARD
# ============================================================

content_rules = (
    content_checked
    .select(
        F.lit("content").alias("entity"),
        F.explode_outer("failed_rule_ids").alias("rule_id")
    )
)

users_rules = (
    users_checked
    .select(
        F.lit("users").alias("entity"),
        F.explode_outer("failed_rule_ids").alias("rule_id")
    )
)

subscription_rules = (
    subs_checked
    .select(
        F.lit("subscriptions").alias("entity"),
        F.explode_outer("failed_rule_ids").alias("rule_id")
    )
)

session_rules = (
    sessions_checked
    .select(
        F.lit("sessions").alias("entity"),
        F.explode_outer("failed_rule_ids").alias("rule_id")
    )
)

# Combine all failed rules
scorecard = (
    content_rules
    .unionByName(users_rules)
    .unionByName(subscription_rules)
    .unionByName(session_rules)
)

# Count failures
scorecard = (
    scorecard
    .filter(F.col("rule_id").isNotNull())
    .groupBy("entity", "rule_id")
    .count()
)

# ============================================================
# APPROVED RULES
# ============================================================

approved_rules = spark.createDataFrame(
    [
        ("content", "DQ-CNT-001"),
        ("users", "DQ-USR-001"),
        ("subscriptions", "DQ-SUB-001"),
        ("sessions", "DQ-SES-001"),
        ("sessions", "DQ-SES-002"),
        ("sessions", "DQ-SES-003"),
        ("sessions", "DQ-SES-004"),
        ("sessions", "DQ-SES-005"),
    ],
    ["entity", "rule_id"]
)

# ============================================================
# FINAL SCORECARD
# ============================================================

scorecard = (
    approved_rules
    .join(
        scorecard,
        ["entity", "rule_id"],
        "left"
    )
    .withColumn(
        "count",
        F.coalesce(F.col("count"), F.lit(0))
    )
    .orderBy("entity", "rule_id")
)

display(scorecard)

entity,rule_id,count
content,DQ-CNT-001,10
sessions,DQ-SES-001,0
sessions,DQ-SES-002,0
sessions,DQ-SES-003,0
sessions,DQ-SES-004,0
sessions,DQ-SES-005,0
subscriptions,DQ-SUB-001,225
users,DQ-USR-001,0


## 10. Multi-failure evidence

A record that fails more than one rule remains **one physical Quarantine row**. The temporary exploded view below is only for inspection.


In [0]:
multi_failure = sessions_checked.filter(F.size("failed_rule_ids") > 1)

display(
    multi_failure.select(
        session_id,
        "record_key",
        "dq_status",
        "failed_rule_ids",
        "failure_reasons",
        "severity"
    ).limit(20)
)


session_id,record_key,dq_status,failed_rule_ids,failure_reasons,severity


## 11. Mandatory reconciliation

For each entity:

**Candidate physical rows = Trusted physical rows + Quarantine physical rows**

The variance must be 0. Trusted and Quarantine must also be disjoint by `record_key`.


In [0]:
# ============================================================
# CANDIDATE -> TRUSTED + QUARANTINE RECONCILIATION
# ============================================================

from pyspark.sql import functions as F


def reconciliation(candidate_table, trusted_table, quarantine_table):

    # Read tables
    candidate = spark.table(candidate_table)
    trusted = spark.table(trusted_table)
    quarantine = spark.table(quarantine_table)

    # --------------------------------------------------------
    # 1. Count physical rows
    # --------------------------------------------------------

    candidate_rows = candidate.count()
    trusted_rows = trusted.count()
    quarantine_rows = quarantine.count()

    # --------------------------------------------------------
    # 2. Check Trusted + Quarantine = Candidate
    # --------------------------------------------------------

    row_variance = (
        candidate_rows
        - trusted_rows
        - quarantine_rows
    )

    # --------------------------------------------------------
    # 3. Check record_key overlap
    #    Only Trusted and Quarantine need record_key
    # --------------------------------------------------------

    if (
        "record_key" in trusted.columns
        and "record_key" in quarantine.columns
    ):

        trusted_keys = (
            trusted
            .select("record_key")
            .filter(F.col("record_key").isNotNull())
            .distinct()
        )

        quarantine_keys = (
            quarantine
            .select("record_key")
            .filter(F.col("record_key").isNotNull())
            .distinct()
        )

        overlap = (
            trusted_keys
            .join(
                quarantine_keys,
                "record_key",
                "inner"
            )
            .count()
        )

        output_distinct_keys = (
            trusted_keys
            .union(quarantine_keys)
            .distinct()
            .count()
        )

    else:

        overlap = -1
        output_distinct_keys = -1

    # --------------------------------------------------------
    # 4. Final status
    # --------------------------------------------------------

    if row_variance == 0 and overlap == 0:
        status = "PASS"
    else:
        status = "FAIL"

    return (
        candidate_table,
        candidate_rows,
        trusted_rows,
        quarantine_rows,
        row_variance,
        output_distinct_keys,
        overlap,
        status
    )


# ============================================================
# RUN RECONCILIATION
# ============================================================

rows = [

    reconciliation(
        TABLES["content"],
        OUTPUTS["content_trusted"],
        OUTPUTS["content_quarantine"]
    ),

    reconciliation(
        TABLES["users"],
        OUTPUTS["users_trusted"],
        OUTPUTS["users_quarantine"]
    ),

    reconciliation(
        TABLES["subscriptions"],
        OUTPUTS["subscriptions_trusted"],
        OUTPUTS["subscriptions_quarantine"]
    ),

    reconciliation(
        TABLES["sessions"],
        OUTPUTS["sessions_trusted"],
        OUTPUTS["sessions_quarantine"]
    )
]


# ============================================================
# CREATE RECONCILIATION DATAFRAME
# ============================================================

recon = spark.createDataFrame(
    rows,
    [
        "candidate_table",
        "candidate_rows",
        "trusted_rows",
        "quarantine_rows",
        "row_variance",
        "output_distinct_keys",
        "trusted_quarantine_overlap",
        "reconciliation_status"
    ]
)


# ============================================================
# DISPLAY RESULT
# ============================================================

display(recon)

candidate_table,candidate_rows,trusted_rows,quarantine_rows,row_variance,output_distinct_keys,trusted_quarantine_overlap,reconciliation_status
workspace.default.silver_content_catalog,2990,2980,10,0,2990,0,PASS
workspace.default.silver_users,24975,24975,0,0,24975,0,PASS
workspace.default.silver_subscriptions,34980,34755,225,0,34980,0,PASS
workspace.default.silver_sessions,249701,249701,0,0,249701,0,PASS


## 12. Session join-multiplication guard

The Team 04 rulebook requires a check that enrichment does not multiply physical session rows. The session count and distinct session ID count must remain stable, and exactly one covering subscription must remain for a valid session.


In [0]:
candidate_session_count = spark.table(TABLES["sessions"]).count()
candidate_session_distinct = spark.table(TABLES["sessions"]).select(session_id).dropDuplicates().count()

checked_session_count = sessions_checked.count()
checked_session_distinct = sessions_checked.select(session_id).dropDuplicates().count()

join_guard = spark.createDataFrame([
    ("Candidate", candidate_session_count, candidate_session_distinct),
    ("After DQ enrichment", checked_session_count, checked_session_distinct),
], ["stage","physical_rows","distinct_session_ids"])

display(join_guard)

if candidate_session_count != checked_session_count:
    raise ValueError("JOIN MULTIPLICATION GUARD FAILED: physical session row count changed.")


stage,physical_rows,distinct_session_ids
Candidate,249701,249701
After DQ enrichment,249701,249701


## 13. Inspect failed records

Use these views to capture `week06_dq_results.png` and `week06_failed_records_sample.png`.

The evidence should show actual Databricks results, not invented counts.


In [0]:
display(
    sessions_quarantine.select(
        session_id,
        "record_key",
        "dq_status",
        "failed_rule_ids",
        "failure_reasons",
        "severity",
        "affected_field",
        "dq_run_id",
        "dq_checked_ts",
        "source_file",
        "batch_id",
        "quarantined_at",
        "rework_status"
    ).limit(25)
)


session_id,record_key,dq_status,failed_rule_ids,failure_reasons,severity,affected_field,dq_run_id,dq_checked_ts,source_file,batch_id,quarantined_at,rework_status


In [0]:
%sql
SELECT 'trusted_content' AS table_name, COUNT(*) AS rows FROM trusted_content
UNION ALL
SELECT 'quarantine_content', COUNT(*) FROM quarantine_content
UNION ALL
SELECT 'trusted_users', COUNT(*) FROM trusted_users
UNION ALL
SELECT 'quarantine_users', COUNT(*) FROM quarantine_users
UNION ALL
SELECT 'trusted_subscriptions', COUNT(*) FROM trusted_subscriptions
UNION ALL
SELECT 'quarantine_subscriptions', COUNT(*) FROM quarantine_subscriptions
UNION ALL
SELECT 'trusted_sessions', COUNT(*) FROM trusted_sessions
UNION ALL
SELECT 'quarantine_sessions', COUNT(*) FROM quarantine_sessions
ORDER BY table_name;


table_name,rows
quarantine_content,10
quarantine_sessions,0
quarantine_subscriptions,225
quarantine_users,0
trusted_content,2980
trusted_sessions,249701
trusted_subscriptions,34755
trusted_users,24975


## 14. Controlled rerun / replay evidence

Do **not** edit Quarantine to make a row pass.

For a real correction/replay:
1. select one quarantined record;
2. explain every failed rule;
3. correct the earliest wrong upstream source/transformation;
4. rerun Bronze → Silver Candidate → applicable DQ rules;
5. verify the new route;
6. retain the original Quarantine evidence;
7. reconcile again.

The cell below records the selected record for investigation without changing any table.


In [0]:
# Select one real quarantined session for replay investigation.
replay_candidate = (
    sessions_quarantine
    .select(
        session_id,
        "record_key",
        "failed_rule_ids",
        "failure_reasons",
        "severity",
        "dq_run_id"
    )
    .limit(1)
)

display(replay_candidate)


session_id,record_key,failed_rule_ids,failure_reasons,severity,dq_run_id


## 15. Week 6 completion checklist

- [x] Eight approved rule IDs represented in the notebook.
- [x] Content, users and subscriptions evaluated before session reference checks.
- [x] All applicable session failures retained together.
- [x] Trusted and Quarantine Delta tables created for four entities.
- [x] Rule-failure scorecard included.
- [x] Candidate/Trusted/Quarantine reconciliation included.
- [x] Session join-multiplication guard included.
- [ ] Approved dictionary values and prohibited-identifying-detail patterns must be populated from the Team 04 project specification where not supplied in the Week-06 rule PDF.
- [ ] Execute in Databricks and capture the required evidence screenshots.
- [ ] Complete the real correction/replay using an actual quarantined record.

**Important:** Do not claim PASS counts, failure counts, screenshots, or replay results until the notebook has been executed in your Databricks workspace.
